# 실습 4: Feature Store & 배치 추론

**목표**: Unity Catalog Feature Store로 피처를 중앙 관리하고, 학습된 모델로 대규모 배치 예측을 수행합니다.

**배울 것**:
- Feature Store에 피처 테이블 생성 및 조회
- Feature Store 기반 모델 학습
- Spark UDF를 이용한 분산 배치 추론

**소요 시간**: ~40분

## Step 1: Feature Engineering — 피처 테이블 생성

In [0]:
from pyspark.sql import functions as F
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

CATALOG = "3dt005_databricks"
SCHEMA = "wine"

# 원본 데이터 로드
wine_raw = spark.table(f"{CATALOG}.{SCHEMA}.wine_quality_lab")

# 고유 키(primary key) 추가 — Feature Store 필수!
wine_with_id = wine_raw.withColumn("wine_id", F.monotonically_increasing_id())

display(wine_with_id.limit(5))

fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality,is_good_quality,wine_id
7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.001,3.0,0.45,8.8,6,0,0
6.3,0.3,0.34,1.6,0.049,14.0,132.0,0.994,3.3,0.49,9.5,6,0,1
8.1,0.28,0.4,6.9,0.05,30.0,97.0,0.9951,3.26,0.44,10.1,6,0,2
7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.4,9.9,6,0,3
7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.4,9.9,6,0,4


In [0]:
# 피처 엔지니어링: 새로운 피처 생성
wine_features = wine_with_id.select(
    "wine_id",
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
    # 파생 피처들
    (F.col("free_sulfur_dioxide") / F.col("total_sulfur_dioxide")).alias("free_sulfur_ratio"),
    (F.col("fixed_acidity") * F.col("alcohol")).alias("acidity_alcohol_interaction"),
    # NaN 방지: alcohol이 0인 경우 대비
    F.when(F.col("alcohol") != 0, F.col("residual_sugar") / F.col("alcohol"))
     .otherwise(0.0).alias("sugar_alcohol_ratio"),
).fillna(0)  # NaN을 0으로 채움

display(wine_features.limit(5))

wine_id,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,free_sulfur_ratio,acidity_alcohol_interaction,sugar_alcohol_ratio
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.001,3.0,0.45,8.8,0.2647058823529412,61.60000000000001,2.352272727272727
1,6.3,0.3,0.34,1.6,0.049,14.0,132.0,0.994,3.3,0.49,9.5,0.10606060606060606,59.85,0.16842105263157894
2,8.1,0.28,0.4,6.9,0.05,30.0,97.0,0.9951,3.26,0.44,10.1,0.30927835051546393,81.80999999999999,0.6831683168316832
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.4,9.9,0.25268817204301075,71.28,0.8585858585858586
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.4,9.9,0.25268817204301075,71.28,0.8585858585858586


## Step 2: Feature Store 테이블 등록

In [0]:
# Feature Store 테이블 생성
FEATURE_TABLE = f"{CATALOG}.{SCHEMA}.wine_features_lab"

try:
    fe.create_table(
        name=FEATURE_TABLE,
        primary_keys=["wine_id"],
        df=wine_features,
        description="와인 품질 예측을 위한 피처 테이블 (3기 실습)",
    )
    print(f"✅ Feature Store 테이블 생성 완료: {FEATURE_TABLE}")
except Exception as e:
    if "already exists" in str(e).lower():
        # 이미 존재하면 덮어쓰기
        fe.write_table(name=FEATURE_TABLE, df=wine_features, mode="overwrite")
        print(f"✅ Feature Store 테이블 업데이트 완료: {FEATURE_TABLE}")
    else:
        raise e

✅ Feature Store 테이블 생성 완료: 3dt005_databricks.wine.wine_features_lab


💡 **Feature Store UI에서 확인해보세요!**
- 왼쪽 메뉴 → Features → 방금 생성된 테이블 클릭
- 피처별 통계, 리니지(계보) 정보를 확인할 수 있습니다

## Step 3: Feature Store에서 피처 조회하여 모델 학습

In [0]:
from databricks.feature_engineering import FeatureLookup
import mlflow

# 라벨 데이터 (wine_id + target만 포함)
labels_df = wine_with_id.select("wine_id", "is_good_quality")

# Feature Lookup 정의
feature_lookups = [
    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key="wine_id",
    )
]

# 학습 데이터셋 생성 (Feature Store에서 자동으로 피처를 조회!)
# exclude_columns: wine_id는 조회 키일 뿐, 학습 피처가 아님!
training_set = fe.create_training_set(
    df=labels_df,
    feature_lookups=feature_lookups,
    label="is_good_quality",
    exclude_columns=["wine_id"],
)

training_df = training_set.load_df()
display(training_df.limit(5))

fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,free_sulfur_ratio,acidity_alcohol_interaction,sugar_alcohol_ratio,is_good_quality
7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.001,3.0,0.45,8.8,0.2647058823529412,61.60000000000001,2.352272727272727,0
6.3,0.3,0.34,1.6,0.049,14.0,132.0,0.994,3.3,0.49,9.5,0.10606060606060606,59.85,0.16842105263157894,0
7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.4,9.9,0.25268817204301075,71.28,0.8585858585858586,0
8.1,0.28,0.4,6.9,0.05,30.0,97.0,0.9951,3.26,0.44,10.1,0.30927835051546393,81.80999999999999,0.6831683168316832,0
7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.4,9.9,0.25268817204301075,71.28,0.8585858585858586,0


In [0]:
# 모델 학습 + Feature Store 연동 로깅
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from mlflow.models import infer_signature

mlflow.set_experiment("/Users/" + spark.sql("SELECT current_user()").first()[0] + "/wine_feature_store_lab")

# Pandas 변환
train_pdf = training_df.toPandas()
X = train_pdf.drop(["is_good_quality"], axis=1)
y = train_pdf["is_good_quality"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run(run_name="feature_store_rf"):
    rf = RandomForestClassifier(n_estimators=150, max_depth=7, random_state=42)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    mlflow.log_metric("f1_score", f1)

    # Feature Store 연동 모델 로깅
    fe.log_model(
        model=rf,
        artifact_path="model",
        flavor=mlflow.sklearn,
        training_set=training_set,
        registered_model_name=f"{CATALOG}.{SCHEMA}.wine_quality_model_lab",
    )

    print(f"✅ Feature Store 연동 모델 학습 완료! F1: {f1:.4f}")

/databricks/python/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


Uploading artifacts:   0%|          | 0/14 [00:00<?, ?it/s]

Registered model '3dt005_databricks.wine.wine_quality_model_lab' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/14 [00:00<?, ?it/s]

Created version '2' of model '3dt005_databricks.wine.wine_quality_model_lab'.


✅ Feature Store 연동 모델 학습 완료! F1: 0.4954


## Step 4: 배치 추론 — Spark UDF로 대용량 예측

In [0]:
# 새로운 데이터가 들어왔다고 가정
# (실제로는 같은 데이터를 사용하지만, 실무에서는 새로운 배치 데이터)
new_data = wine_with_id.select(F.col("wine_id").cast("long")).limit(100)

# 등록된 모델의 최신 버전 자동 조회
from mlflow import MlflowClient
client = MlflowClient(registry_uri="databricks-uc")
model_name = f"{CATALOG}.{SCHEMA}.wine_quality_model_lab"
versions = client.search_model_versions(f"name='{model_name}'")
if versions:
    latest_version = max(int(v.version) for v in versions)
    print(f"✅ 모델 '{model_name}' 최신 버전: {latest_version}")
else:
    latest_version = 1
    print(f"⚠️ 모델 버전을 찾을 수 없어 기본값 1 사용")

# Feature Store 기반 배치 추론
# wine_id만 있으면 자동으로 Feature Store에서 피처를 조회하여 예측!
try:
    predictions = fe.score_batch(
        model_uri=f"models:/{model_name}/{latest_version}",
        df=new_data,
    )
    display(predictions.select("wine_id", "prediction"))
except Exception as e:
    print(f"⚠️ score_batch 오류: {e}")
    print("→ Feature Store 테이블과 모델을 처음부터 다시 생성해주세요 (Step 1부터 재실행)")
    print("→ 또는 아래 Spark UDF 방식을 사용해주세요")

✅ 모델 '3dt005_databricks.wine.wine_quality_model_lab' 최신 버전: 2


2026/03/24 05:09:25 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2026/03/24 05:09:26 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


wine_id,prediction
26,0.0
29,1.0
65,0.0
19,0.0
54,0.0
0,0.0
22,0.0
7,0.0
77,0.0
34,0.0


### Spark UDF 방식 배치 추론 (대용량 데이터)

In [0]:
# sklearn 모델을 직접 Spark pandas_udf로 변환 — 분산 환경에서 수백만 건도 OK!
import pandas as pd
from pyspark.sql.types import IntegerType

# Step 3에서 학습한 rf 모델을 직접 사용 (Feature Store 래퍼 없이)
feature_cols = [c for c in X.columns]

@F.pandas_udf(IntegerType())
def predict_udf(*cols):
    X_input = pd.concat(cols, axis=1)
    X_input.columns = feature_cols
    X_input = X_input.fillna(0)
    return pd.Series(rf.predict(X_input))

# 전체 데이터에 대해 예측 수행
result_df = (
    wine_features
    .withColumn("prediction", predict_udf(*[F.col(c) for c in feature_cols]))
)

display(result_df.select("wine_id", "prediction").limit(10))

wine_id,prediction
0,0
1,0
2,0
3,0
4,0
5,0
6,0
7,0
8,0
9,0


In [0]:
# 추론 결과를 Delta 테이블로 저장
result_df.select("wine_id", "prediction").write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.wine_predictions_lab"
)

print(f"✅ 배치 추론 결과 저장 완료: {CATALOG}.{SCHEMA}.wine_predictions_lab")
print(f"   총 {result_df.count():,}건 예측 완료")

✅ 배치 추론 결과 저장 완료: 3dt005_databricks.wine.wine_predictions_lab
   총 4,898건 예측 완료


## 🎯 핵심 정리

| 단계 | 코드 | 설명 |
|------|------|------|
| **피처 테이블 생성** | `fe.create_table()` | 피처를 중앙 관리 |
| **학습 데이터셋** | `fe.create_training_set()` | 피처 자동 조회 |
| **모델 로깅** | `fe.log_model()` | Feature Store 연동 저장 |
| **배치 추론** | `fe.score_batch()` | ID만으로 자동 예측 |
| **Spark UDF** | `mlflow.pyfunc.spark_udf()` | 대용량 분산 추론 |

💡 **핵심 가치**: Feature Store를 사용하면 **피처 재사용** + **학습-서빙 일관성** + **리니지 추적**이 자동으로 됩니다!

---
**Day 3 실습 완료!** 내일은 모델 배포와 LLM을 다룹니다.